In [ ]:
# bvbrc_processing.ipynb
# Purpose: Prepare clean BV-BRC E. coli ciprofloxacin-resistant dataset
# Outputs:
#   - bvbrc_genomes.tsv       (per-genome metadata)
#   - bvbrc_contigs.csv       (per-contig: id, sequence, length)

import re
import pandas as pd
from pathlib import Path
from Bio import SeqIO

In [2]:
# ────────────────────────────────────────────────
#  CONFIG / PATHS
# ────────────────────────────────────────────────

BV_BRC_TSV_PATH    = "ecoli_cipro_500_ids.tsv"           # from p3-get-drug-genomes
METADATA_PATH      = "ecoli_metadata_full.tsv"           # from p3-get-genome-data
FASTA_DIR          = Path("fasta_files")                 # where you saved the FASTAs

OUTPUT_GENOMES     = "bvbrc_genomes.tsv"
OUTPUT_CONTIGS     = "bvbrc_contigs.csv"

In [3]:
# ────────────────────────────────────────────────
#  1. Load phenotype + basic info
# ────────────────────────────────────────────────

print("Loading phenotype data...")
drug_df = pd.read_csv(BV_BRC_TSV_PATH, sep='\t', dtype=str)

drug_df = drug_df.rename(columns={
    'genome_drug.genome_id':          'genome_id',
    'genome_drug.antibiotic':         'antibiotic',
    'genome_drug.resistant_phenotype':'resistant_phenotype'
})

drug_df = drug_df[['genome_id', 'antibiotic', 'resistant_phenotype']]
drug_df['genome_id'] = drug_df['genome_id'].str.strip()

print(f"→ {len(drug_df)} resistant phenotype records")

Loading phenotype data...
→ 500 resistant phenotype records


In [4]:
# ────────────────────────────────────────────────
#  2. Load richer metadata
# ────────────────────────────────────────────────

print("\nLoading genome metadata...")
meta_df = pd.read_csv(METADATA_PATH, sep='\t', dtype=str)

meta_df['genome_id'] = meta_df['genome_id'].str.strip()
meta_df['genome_length'] = pd.to_numeric(meta_df['genome_length'], errors='coerce')

keep_cols = [
    'genome_id', 'genome_name', 'taxon_id', 'taxon_lineage_names',
    'genome_length', 'antimicrobial_resistance', 'antimicrobial_resistance_evidence'
]

meta_df = meta_df[keep_cols].copy()


Loading genome metadata...


In [5]:
# ────────────────────────────────────────────────
#  3. Merge → base genome table
# ────────────────────────────────────────────────

print("\nMerging phenotype + metadata...")
genomes_df = pd.merge(
    drug_df,
    meta_df,
    on='genome_id',
    how='left',
    validate='many_to_one'
)

print(f"→ {len(genomes_df)} rows after merge")


Merging phenotype + metadata...
→ 500 rows after merge


In [6]:
# ────────────────────────────────────────────────
#  4. Parse FASTA files → contig table
# ────────────────────────────────────────────────

print("\nParsing contigs from FASTA directory...")
contig_list = []

global_order = 0

for fasta_path in FASTA_DIR.glob("*_contigs.fasta"):
    genome_id = fasta_path.stem.replace("_contigs", "").strip()
    order = 0
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        header = record.id.strip()
        contig_id = header.split()[0]
        
        contig_list.append({
            'genome_id': genome_id,
            'contig_id': contig_id,
            'contig_length': len(record.seq),
            'contig_sequence': str(record.seq).upper(),
            'fasta_order': order,
        })
        order += 1

contigs_df = pd.DataFrame(contig_list)

print(f"→ {len(contigs_df):,} contigs from {contigs_df['genome_id'].nunique()} genomes")


Parsing contigs from FASTA directory...
→ 73,423 contigs from 500 genomes


In [7]:
# ────────────────────────────────────────────────
#  5. Calculate assembled length per genome
# ────────────────────────────────────────────────

print("\nCalculating assembled length from contigs...")
assembled_lengths = (
    contigs_df
    .groupby('genome_id', as_index=False)
    ['contig_length']
    .sum()
    .rename(columns={'contig_length': 'assembled_length'})
)

genomes_df = pd.merge(
    genomes_df,
    assembled_lengths,
    on='genome_id',
    how='left'
)

genomes_df['length_match'] = genomes_df['genome_length'] == genomes_df['assembled_length']
genomes_df['length_match'] = genomes_df['length_match'].fillna(False)

print("Length match summary:")
print(genomes_df['length_match'].value_counts(dropna=False))


Calculating assembled length from contigs...
Length match summary:
length_match
True    500
Name: count, dtype: int64


In [8]:
# Filter: keep only genomes where lengths match
genomes_df = genomes_df[genomes_df['length_match']].copy()
print(f"\n→ Keeping {len(genomes_df)} genomes after length filter")

# Drop temporary column
genomes_df = genomes_df.drop(columns=['length_match', 'assembled_length'])


→ Keeping 500 genomes after length filter


In [9]:
# ────────────────────────────────────────────────
#  6. Final contig table — only for kept genomes
# ────────────────────────────────────────────────

print("\nFiltering contigs to match kept genomes...")
contigs_df = contigs_df[contigs_df['genome_id'].isin(genomes_df['genome_id'])].copy()

print(f"→ {len(contigs_df):,} contigs remain")


Filtering contigs to match kept genomes...
→ 73,423 contigs remain


In [10]:
genomes_df.head()

,genome_id,antibiotic,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence
0,562.8523,ciprofloxacin,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN
1,562.7542,ciprofloxacin,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN
2,562.98417,ciprofloxacin,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel
3,562.140842,ciprofloxacin,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel
4,562.135581,ciprofloxacin,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN


In [11]:
genomes_df = genomes_df.loc[:, ~genomes_df.columns.duplicated()]
genomes_df.head()

,genome_id,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence
0,562.8523,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN
1,562.7542,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN
2,562.98417,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel
3,562.140842,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel
4,562.135581,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN


In [12]:
contigs_df.head()

,genome_id,contig_id,contig_length,contig_sequence,fasta_order
0,562.140899,562.140899.con.0143,593,TGCTCCCGCTGCCCGGGCGGCAGTTCAGCCTCCGCGAACGACAGCA...,0
1,562.140899,562.140899.con.0148,544,TGTTGCCCGTATTGTGGCGCTGTCGACCCCTTCGGTTATTACCGGA...,1
2,562.140899,562.140899.con.0151,533,ACATTGTCAAGATGAACAGCATGCATTGTAACAGTCTGCAGTTTAT...,2
3,562.140899,562.140899.con.0164,457,AACTGACTGTGGCGCGTATAATTTCTCTGCGTTAATTTTTTTGTCG...,3
4,562.140899,562.140899.con.0009,137535,CTTATACTCCCACATAAGCCAGATTCAACAGCGAATACGTCTTCCC...,4


In [13]:
# ────────────────────────────────────────────────
#  7. Save outputs
# ────────────────────────────────────────────────

print("\nSaving...")

genomes_df.to_csv(OUTPUT_GENOMES, sep='\t', index=False)
print(f"  {OUTPUT_GENOMES:<28} ({len(genomes_df):,} genomes)")

contigs_df.to_csv(OUTPUT_CONTIGS, index=False)
print(f"  {OUTPUT_CONTIGS:<28} ({len(contigs_df):,} contigs)")

print("\nDone.")
print("Next steps:")
print(" - Use bvbrc_genomes.tsv for genome-level statistics")
print(" - Use bvbrc_contigs.csv + gyrA/parC/parE features for gene extraction")


Saving...
  bvbrc_genomes.tsv            (500 genomes)
  bvbrc_contigs.csv            (73,423 contigs)

Done.
Next steps:
 - Use bvbrc_genomes.tsv for genome-level statistics
 - Use bvbrc_contigs.csv + gyrA/parC/parE features for gene extraction
